# Reproducible pipeline

The project detects `own`, `leak`, and `third` speech on both channels of an 8 kHz stereo telephone recording and removes leakage. This notebook covers data synthesis, optional training, validation, and inference.

The existing model weights are used by default. Set `TRAIN = True` to retrain the detector and cleaner on the bundled Russian speech data.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import soundfile as sf

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

TRAIN = False
DIALOGUE_DURATION = 3.0
SEED = 42
ROOT

## 1. Data

Training and test dialogues are synthesized from disjoint speaker folders. The synthesizer creates the physical scenarios and ground-truth intervals used by the detector and cleaner.

In [ ]:
from src.data_loader import RealSpeechLoader
from src.synthesizer import TelephoneDialogueSynthesizer

loader = RealSpeechLoader()
synthesizer = TelephoneDialogueSynthesizer(seed=SEED)
speakers = loader.discover_speakers()
print('Speakers:', list(speakers))

speaker_a, speaker_b, speaker_third = loader.sample_dialogue_speakers(
    'test', russian_prob=1.0
)
pool = loader.build_utterance_pool(speaker_a, speaker_b, speaker_third, max_per_speaker=4)
example = synthesizer.synthesize_dialogue(
    utterance_pool=pool,
    duration_sec=DIALOGUE_DURATION,
    scenario='normal',
)
print(example.audio.shape, example.sample_rate, example.ground_truth_intervals.keys())

## 2. Training

The commands call the maintained training entry points and do not alter model architecture or preprocessing. Small values are useful for a smoke test; the defaults in the scripts are intended for a full training run.

In [ ]:
def run(command):
    print(' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

if TRAIN:
    run([sys.executable, 'scripts/train_detector.py'])
    run([sys.executable, 'scripts/train_cleaner.py'])
else:
    print('Using models/detector_weights.pt and models/cleaner_weights.pt')

## 3. Validation

Run the automated tests for the DSP pipeline, model interfaces, cleaner behavior, and CPU real-time constraint. The full scenario evaluation is optional because it synthesizes a larger test set.

In [ ]:
run([sys.executable, '-m', 'pytest', 'tests'])

In [ ]:
run([sys.executable, 'evaluate.py', '--num_per_scenario', '1', '--duration_sec', '3.0', '--seed', str(SEED)])

## 4. Inference

The same saved stereo WAV is passed through both public command-line interfaces. The JSON output contains channel intervals; the WAV output contains the cleaned stereo signal.

In [ ]:
input_path = ROOT / 'notebooks' / 'pipeline_input.wav'
json_path = ROOT / 'notebooks' / 'pipeline_predictions.json'
clean_path = ROOT / 'notebooks' / 'pipeline_cleaned.wav'
sf.write(input_path, example.audio.T, example.sample_rate, subtype='PCM_16')
run([sys.executable, 'detect.py', str(input_path), str(json_path)])
run([sys.executable, 'clean.py', str(input_path), str(clean_path)])
print(json.dumps(json.loads(json_path.read_text()), ensure_ascii=False, indent=2)[:1500])
print('Cleaned file:', clean_path)